# 🏔️ RescueNex AI Engine: Himalayan Landslide Early Warning ML Pipeline

**Project:** RescueNex Early Warning System  
**Objective:** Multi-parameter geotechnical & hydrometeorological landslide susceptibility classification, Landslide Stability Index (LSI) regression, and Time-to-Failure (TTF) estimation.  
**Methodology:** Mohr-Coulomb limit equilibrium, Caine intensity-duration empirical rainfall thresholds, and Saito tertiary creep kinetics.

In [ ]:
import os
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score, mean_absolute_error, r2_score

print("Libraries loaded successfully.")

## 1. Load and Inspect Geotechnical Sensor Dataset

In [ ]:
dataset_path = os.path.join('..', 'dataset', 'landslide_sensor_data.csv')
df = pd.read_csv(dataset_path)
print(f"Total records: {len(df)}")
df.head()

In [ ]:
df.info()

## 2. Exploratory Data Analysis & Class Distribution

In [ ]:
class_counts = df['riskLevel'].value_counts()
print(class_counts)

plt.figure(figsize=(8, 4))
colors = {'LOW': '#22c55e', 'ADVISORY': '#eab308', 'WARNING': '#f97316', 'CRITICAL': '#ef4444'}
class_counts.plot(kind='bar', color=[colors[c] for c in class_counts.index], edgecolor='black')
plt.title('Distribution of Landslide Risk Categories')
plt.xlabel('Risk Level')
plt.ylabel('Sensor Record Count')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

## 3. Feature Correlation Analysis

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns
corr = df[numeric_cols].corr()
print("Correlation with Landslide Stability Index (lsiScore):")
print(corr['lsiScore'].sort_values(ascending=False))

## 4. Train-Test Split and Feature Scaling

In [ ]:
FEATURE_COLUMNS = [
    'soilMoisturePct', 'tiltMagnitude', 'tiltDelta', 'accelRms',
    'vibrationDetected', 'rain1hIntensityMm', 'rain24hCumulativeMm',
    'antecedentRain7dMm', 'slopeAngleDeg', 'temperatureC', 'humidityPct'
]

X = df[FEATURE_COLUMNS]
y_class = df['riskLevel']
y_lsi = df['lsiScore']

X_train, X_test, y_class_train, y_class_test, y_lsi_train, y_lsi_test = train_test_split(
    X, y_class, y_lsi, test_size=0.20, random_state=42, stratify=y_class
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print(f"X_train shape: {X_train.shape}, X_test shape: {X_test.shape}")

## 5. Model 1: Multi-Class Landslide Risk Classifier (Gradient Boosting)

In [ ]:
clf = GradientBoostingClassifier(n_estimators=150, learning_rate=0.1, max_depth=4, random_state=42)
clf.fit(X_train_scaled, y_class_train)

y_pred_class = clf.predict(X_test_scaled)
acc = accuracy_score(y_class_test, y_pred_class)
f1 = f1_score(y_class_test, y_pred_class, average='weighted')

print(f"Classifier Accuracy: {acc*100:.2f}%")
print(f"Weighted F1-Score: {f1:.4f}\n")
print(classification_report(y_class_test, y_pred_class))

## 6. Model 2: Landslide Stability Index (LSI) Regressor

In [ ]:
reg = GradientBoostingRegressor(n_estimators=120, learning_rate=0.1, max_depth=4, random_state=42)
reg.fit(X_train_scaled, y_lsi_train)

y_pred_lsi = reg.predict(X_test_scaled)
mae = mean_absolute_error(y_lsi_test, y_pred_lsi)
r2 = r2_score(y_lsi_test, y_pred_lsi)

print(f"LSI Regressor MAE: {mae:.2f} points")
print(f"LSI Regressor R² Score: {r2:.4f}")

## 7. Feature Importance & Interpretation

In [ ]:
importances = clf.feature_importances_
indices = np.argsort(importances)[::-1]

plt.figure(figsize=(10, 5))
plt.title('Feature Importances for Landslide Risk Prediction')
plt.bar([FEATURE_COLUMNS[i] for i in indices], importances[indices], color='#f59e0b', edgecolor='black')
plt.xticks(rotation=45, ha='right')
plt.ylabel('Relative Importance')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

## 8. Exporting Production Model Weights

In [ ]:
weights_dir = os.path.join('..', '..', 'app', 'models', 'weights')
os.makedirs(weights_dir, exist_ok=True)

joblib.dump(scaler, os.path.join(weights_dir, 'scaler.joblib'))
joblib.dump(clf, os.path.join(weights_dir, 'landslide_classifier.joblib'))
joblib.dump(reg, os.path.join(weights_dir, 'landslide_lsi_regressor.joblib'))
print("All models saved and ready for the FastAPI AI microservice.")